### AI Agents and Big Data Analytics - Better Together
Up to 80% of the data that enterprises generate is unstructured and semi-structured data. Your AI Agents will provide richer insights by querying TBs of data stored in S3 data lake.<br><br>
Do AI developers need to learn data engineering skills such as distributed spark processing, metadata catalogs, complex distributed SQL queries, NoSQL queries etc to query TBs and PBs of data in data lakes?<br><br>
Fortunately, the answer is no. You just need to know the basics. You can use Strands Agents, built-in strands tools such as use_aws, and MCP server such as AWS Data Processing MCP Server to accomplish what most data engineers can, while writing few lines of code.<br><br>
This notebook will help you bridge the gap between a data engineer who knows how to use AWS Glue, Amazon Athena with an AI engineer who knows how to use Strands SDK, Amazon Bedrock. You will be able to do sophisticated analytics with natural language questions.<br>


### GOAL - Perform big data analytics using AI Agents without writing big data analytics code. <br>
The AI Agent will use built-in strands tools and MCP server to get the job done. 

### In this Notebook
You will learn how to use AI Agents to discover metadata of big-data stored in parquet files using Agent tools and MCP server that in turn uses AWS Glue and Amazon Athena.<br><br>
We will use NYC Taxi and Limousine Commission (TLC) data

### Environment Setup
Install required dependencies for the notebook including Strands SDK, AWS SDK, and MCP client libraries.

In [1]:
# Install all required packages from requirements.txt
!pip install -r requirements.txt --quiet
!pip install --upgrade strands-agents --quiet

In [2]:
# Setup the logging
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%m/%d %H:%M:%S',
    filename='strands_debug.log'
)


### Pre-Requisites
1/ Create an S3 bucket or reuse an existing one.<br>
2/ Create a folder in the S3 bucket so you can upload Parquet files to this folder.<br>
3/ Download the data from NYC Taxi and Limousine Commission (TLC) https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page <br>Expand 2025 and download Yellow Taxi ip Records.<br>
4/ Upload them to the folder in S3 bucket.<br><br>
5/ Specify the following in inputs.txt file in the current folder<br>
---- name of the s3 bucket<br>
---- name of the folder where the parquet files files are uploaded. Ex: s3://my-folder-name/nyc_data_source/<br>
---- name of the aws region. Ex: us-west-2<br>
---- name of the aws role that has permissions for AWS Glue, Amazon Athena, Amazon S3, Amazon Bedrock, Amazon Knowledge Bases<br>
---- name of the S3 folder where Amazon Athena should store the results. Ex: s3://my-folder-name/athena_results/<br>
---- name of the S3 folder where Athena results should be stored<br><br>

6/ You also need run aws configure on your laptop and provide access key ID, key access, default region name<br>
7/ And lastly you need model access to all Anthropic Claude Sonnet Models and Amazon Titan Embeddings model. 

### Import Dependencies and AWS Configuration
Import required libraries and configure AWS settings for the data processing workflow.

In [3]:
# Import required libraries
import os, time, boto3, json
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import use_aws, file_write, file_read, file_write, sleep, python_repl
from datetime import datetime
from pprint import pprint
from pydantic import BaseModel, Field
from typing import Optional, List

# Bypass tool consent for automated execution
os.environ["BYPASS_TOOL_CONSENT"] = "true"
# Specify that if python_repl tool is used, it shouldnt wait for user interaction
os.environ["PYTHON_REPL_INTERACTIVE"] = "False"

model_list = ['deepseek.v3-v1:0', 'qwen.qwen3-coder-30b-a3b-v1:0', 'us.anthropic.claude-3-7-sonnet-20250219-v1:0', 'us.anthropic.claude-sonnet-4-20250514-v1:0', 'openai.gpt-oss-20b-1:0', 'openai.gpt-oss-120b-1:0']

# We will use the following model in Strands Agent
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

## Data Source Configuration

In [4]:
from utils_big_data import read_config_data, print_tokens_costs, load_system_prompt_from_file
# Read the inputs.txt file to extract s3 bucket name, s3 folder, aws_region and other configuration parameters
config_dict = read_config_data()
config_dict


Tool #1: ConfigurationData
s3_bucket_name='chreka-mkhnrsmo' s3_folder_name='s3://chreka-mkhnrsmo/nyc_data_source/' aws_region='us-west-2' aws_role_name='chreka-mkhnrsmo-glue-crawler-role' s3_bucket_for_athena_output='s3://chreka-mkhnrsmo/athena_results/' kb_id=None


{'s3_bucket_name': 'chreka-mkhnrsmo',
 's3_folder_name': 's3://chreka-mkhnrsmo/nyc_data_source/',
 'aws_region': 'us-west-2',
 'aws_role_name': 'chreka-mkhnrsmo-glue-crawler-role',
 's3_bucket_for_athena_output': 's3://chreka-mkhnrsmo/athena_results/',
 'kb_id': None}

The first step is to discover the metadata and catalog it. <br><br>
AWS Glue can help crawl the data, extract the metadata, and create a catalog using AWS Glue Catalog.<br><br>
Let's NOT write code to create AWS Glue jobs. Instead, let's write a SYSTEM PROMPT that will instruct an AI Agent to do this job for us.<br>

### Catalog Data in S3 as Glue Catalog Database and Tables
Create a Strands Agent that uses use_aws strands tool to create AWS Glue crawlers and catalog the S3 data automatically.

In [5]:
# Let's load the system prompt from file
crawl_task = load_system_prompt_from_file("crawl_task_prompt.txt", config_dict=config_dict, s3_folder_name=config_dict['s3_folder_name'], s3_bucket_name=config_dict['s3_bucket_name'])
crawl_task

'I have files in an S3 folder and need you to create a Glue Crawler to crawl them and create/update a Glue Catalog. Use the use_aws tool to complete this task.\n\nConfiguration details: {\'s3_bucket_name\': \'chreka-mkhnrsmo\', \'s3_folder_name\': \'s3://chreka-mkhnrsmo/nyc_data_source/\', \'aws_region\': \'us-west-2\', \'aws_role_name\': \'chreka-mkhnrsmo-glue-crawler-role\', \'s3_bucket_for_athena_output\': \'s3://chreka-mkhnrsmo/athena_results/\', \'kb_id\': None}. Strictly use the specified AWS region specified.\n\nIMPORTANT: \n2. Use the files under s3://chreka-mkhnrsmo/nyc_data_source/.\n\nPlease perform these tasks using the use_aws tool:\n\n1. **Check if database exists**: \n-- If the database already exits and it points to the folder s3://chreka-mkhnrsmo/nyc_data_source/, delete all tables in that database.\n-- If the database does not exist, create it. The database name must include chreka-mkhnrsmo\n\n2. **Always create a glue crawler**:\n   - Do not use an existing one.\n   

In [6]:
# Initialize Claude 3.7 Sonnet model via Bedrock
model = BedrockModel(model_id=model_id, temperature=0.1)  # Low temperature for consistent structured output

# Create the Strands Agent. 
# use_aws tool will create a glue crawler and create glue catalog db and table schema
# sleep tool will help the agent wait until the glue crwler job is finished.
# This task might take a few minutes. You can go to AWS Console and see the Glue Crawler jobs that are being created.
crawl_agent = Agent(model=model, tools=[use_aws, sleep])

crawl_response = crawl_agent(crawl_task)
print(crawl_response)

I'll help you create a Glue Crawler to crawl your S3 files and create/update a Glue Catalog. Let me break this down into steps as you requested.

First, let's check if the database exists and handle it accordingly.
Tool #1: use_aws


╭─ 🚀 List Glue databases to check if one exists for our data ─╮
│                                                              │
│  Service:        glue                                        │
│  Operation:      get_databases                               │
│  Region:         us-west-2                                   │
│  Parameters:     None                                        │
│                                                              │
╰──────────────────────────────────────────────────────────────╯

I see that a database named "chreka-mkhnrsmo-db" already exists. Let's check if it has any tables that need to be deleted.
Tool #2: use_aws


╭─ 🚀 List tables in the existing database ─╮
│                                           │
│  Service:            glue                 │
│  Operation:          get_tables           │
│  Region:             us-west-2            │
│  Parameters:                              │
│    • DatabaseName:   chreka-mkhnrsmo-db   │
│                                           │
╰───────────────────────────────────────────╯

I found a table named "nyc_data_source" in the database. Since we need to delete all tables in the existing database, let's delete this table.
Tool #3: use_aws


...

The crawler is still in the "STOPPING" state. Let's wait for another 15 seconds and check again.
Tool #21: sleep
Let's check the crawler status again:
Tool #22: use_aws


╭───────────── 🚀 Check crawler status ──────────────╮
│                                                    │
│  Service:        glue                              │
│  Operation:      get_crawler                       │
│  Region:         us-west-2                         │
│  Parameters:                                       │
│    • Name:       chreka-mkhnrsmo-nyc-data-crawler  │
│                                                    │
╰────────────────────────────────────────────────────╯

Great! The crawler is now in the "READY" state and has successfully completed its run. Let's verify the results by checking the tables in the database.
Tool #23: use_aws


╭───── 🚀 List tables in the database ─────╮
│                                          │
│  Service:            glue                │
│  Operation:          get_tables          │
│  Region:             us-west-2           │
│  Parameters:                             │
│    • DatabaseName:   chreka-mkhnrsmo-db  │
│                                          │
╰──────────────────────────────────────────╯

Perfect! The crawler has successfully created a table named "nyc_data_source" in the "chreka-mkhnrsmo-db" database. The table contains the schema for the NYC data source files.

Here's the summary of what we've accomplished:

1. We checked if the database "chreka-mkhnrsmo-db" existed, which it did.
2. We deleted the existing table in the database.
3. We deleted the existing crawler and created a new one.
4. We started the crawler and monitored its status until it completed successfully.
5. We verified that the crawler created a table in the database.

Here's the requested JSON response:

```json
{
  "catalog_db_name": "chreka-mkhnrsmo-db",
  "catalog_table_names": ["nyc_data_source"]
}
```

The Glue Crawler has successfully crawled the S3 data at s3://chreka-mkhnrsmo/nyc_data_source/ and created a table in the Glue Data Catalog. You can now use this table for querying the data using services like Athena.Perfect! The crawler has successfully created a table named "nyc_data_source" in th

In [7]:
from utils_big_data import print_tokens_costs

# Let's print the token costs
print_tokens_costs(crawl_response)


{'latencyMs': 86660}
{'inputTokens': 187858, 'outputTokens': 3903, 'totalTokens': 191761}
Time to research = 86.66 seconds
Input Token Costs = $0.563574;
Output Token Costs = $0.058545
Total Token Costs = $0.622119


## Extract Glue Database and Tables Names
Parse the agent response to extract the Glue database name and Table names for use in subsequent queries.

In [8]:
# Strands Agenst allows you to extract information in a structured dictionary output.

# Define the structured template
class GlueDbTableInfo(BaseModel):
    catalog_db_name: str = Field(description="Name of the Glue Catalog Database")
    catalog_table_names: List[str] = Field(description="List of glue table names")

# Use the structured_output method
glue_db_table_info = crawl_agent.structured_output(GlueDbTableInfo, f"Extract Glue Catalog Database name and the list of glue table names")
print(glue_db_table_info)
catalog_db_name = glue_db_table_info.catalog_db_name
catalog_table_names = glue_db_table_info.catalog_table_names


Tool #24: GlueDbTableInfo
catalog_db_name='chreka-mkhnrsmo-db' catalog_table_names=['nyc_data_source']


In [9]:

config_dict['catalog_db_name'] = catalog_db_name
config_dict['catalog_table_names'] = catalog_table_names
config_dict

{'s3_bucket_name': 'chreka-mkhnrsmo',
 's3_folder_name': 's3://chreka-mkhnrsmo/nyc_data_source/',
 'aws_region': 'us-west-2',
 'aws_role_name': 'chreka-mkhnrsmo-glue-crawler-role',
 's3_bucket_for_athena_output': 's3://chreka-mkhnrsmo/athena_results/',
 'kb_id': None,
 'catalog_db_name': 'chreka-mkhnrsmo-db',
 'catalog_table_names': ['nyc_data_source']}

AWS Labs has created many MCP servers for AI Agent developers to consume. One of them is AWS data processing MCP server: https://awslabs.github.io/mcp/servers/aws-dataprocessing-mcp-server <br><br>
This MCP server exposes AI Agent tools to do operations on big data using AWS Glue, Amazon EMR, and Amazon Athena.<br><br>
The best part is that we don't have to know the intricacies of these services other than high level basics.<br><br>
Let's create an MCP client for this MCP server.

## MCP Client Setup
We wll get tools exposed by an MCP server to discover partition columns and keys from the data in S3.<br><br>
Initialize the AWS Data Processing MCP server client to provide AI agents with AWS Glue, EMR, and Athena capabilities.

In [10]:
aws_region = config_dict['aws_region']
# Import MCP client libraries
from mcp import stdio_client, StdioServerParameters
from strands.tools.mcp import MCPClient

# Create MCP client for AWS data processing server
# This provides tools for Glue, EMR, and Athena operations
data_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="uvx",  # Use uvx to run the MCP server
        args= [
            "awslabs.aws-dataprocessing-mcp-server@latest",
            "--allow-write",  # Enable write operations
        ],
        env= {
            "FASTMCP_LOG_LEVEL": "ERROR",  # Minimize logging noise
            "AWS_REGION": aws_region      # Set AWS region
      }
    )
))

## Print the tools offered by the MCP Server
This will help you understand what the tools do. This is extracted by looking at doc strings or tool spec of the tools.<br><br>
Agents send the doc string or the tool spec of the tools to the LLM along with a task.<br>
This helps LLM to reason and decide which tool to use for which task.

In [11]:
# Connect to MCP client and execute task
tools_char_count = 0
tool_count = 0
with data_mcp_client:
    import json
    # Get available tools from the MCP server
    data_tools = data_mcp_client.list_tools_sync()
    # first_tool = data_tools[0]
    # print(dir(first_tool))    
    # Iterate through each tool
    for tool in data_tools:
        print(f"Tool: {tool.tool_name}")
        if hasattr(tool, 'tool_spec'):            
            print(f"Tool: {tool.tool_spec['description']}")
            # print(json.dumps(tool.tool_spec, indent=2)) # uncomment this to see function parameters and what they mean   
            tools_char_count += len(json.dumps(tool.tool_spec))
            tool_count += 1
        print("-" * 50)
print(f"The number of characters in the spec of all the {tool_count} tools = {tools_char_count}")

Tool: manage_aws_glue_databases
Tool: Manage AWS Glue Data Catalog databases with both read and write operations.

        This tool provides operations for managing Glue Data Catalog databases, including creating,
        updating, retrieving, listing, and deleting databases. It serves as the primary mechanism
        for database management within the AWS Glue Data Catalog.

        ## Requirements
        - The server must be run with the `--allow-write` flag for create-database, update-database, and delete-database operations
        - Appropriate AWS permissions for Glue Data Catalog operations

        ## Operations
        - **create-database**: Create a new database in the Glue Data Catalog
        - **delete-database**: Delete an existing database from the Glue Data Catalog
        - **get-database**: Retrieve detailed information about a specific database
        - **list-databases**: List all databases in the Glue Data Catalog
        - **update-database**: Update an existin

### Understand the content that the tables have and what the columns mean. 
Store this in a file and pass it to the agent. This will help agent construct SQL queries properly.


In [12]:
# Get the definition of the table and columns and store it in a file.
table_def_system_prompt = f"""
You are an expert AWS data analyst assistant specializing in querying data stored in S3 data lakes using AWS Glue and Amazon Athena.

## IMPORTANT: 
- For any long running jobs, sleep for 20 seconds and check status until the job is finished. Do not give up until the job actually finishes.
"""
query_in = f"""Query the database {config_dict['catalog_db_name']}, identify the tables in it, identify the columns and their types. Gather more business context of what the database, tables, and the columns are storing by sampling a few rows of data. Then store the database name, table names, their purpose; and column name and its type and purpose in a json file named metadata.json."""

column_def_response = ""
# Get the data processing tools from MCP server
with data_mcp_client:
    data_tools = data_mcp_client.list_tools_sync()
    curated_data_tools = ['manage_aws_athena_query_executions', 'manage_aws_glue_tables']

    # Extract just the tools that we need.
    filtered_tools = [tool for tool in data_tools if tool.tool_name in curated_data_tools]
    filtered_tools += [file_write, file_read, sleep]

    # Pass the system prompt, the LLM we use with bedrock, and all the tools to the agent
    column_def = Agent(system_prompt = table_def_system_prompt, model=model, tools=filtered_tools)
    # Invoke the agent with each of the query
    column_def_response = column_def(query_in)

I'll help you query the database, identify tables, analyze their structure, and gather business context by sampling data. Then I'll store all this information in a metadata.json file as requested.

Let's start by listing the tables in the database "chreka-mkhnrsmo-db":
Tool #1: manage_aws_glue_tables
Now that I've identified the table "nyc_data_source" in the database, let's get more detailed information about this table:
Tool #2: manage_aws_glue_tables
Now, let's sample some data from the table to understand the business context better:
Tool #3: manage_aws_athena_query_executions
I need to specify an output location for the Athena query results. Let's try again with a result configuration:
Tool #4: manage_aws_athena_query_executions
Now let's wait for the query to complete and then get the results:
Tool #5: sleep
Let's check the status of our query:
Tool #6: manage_aws_athena_query_executions
Great! The query has succeeded. Now let's get the results:
Tool #7: manage_aws_athena_query_e

╔═ File Write Operation ═╗
║                        ║
║ Path: metadata.json    ║
║ Size: 3687 characters  ║
║                        ║
╚════════════════════════╝

╔═════════════ Write Successful ═════════════╗
║ File written successfully to metadata.json ║
╚════════════════════════════════════════════╝

I've successfully completed the requested tasks:

1. Queried the database "chreka-mkhnrsmo-db" and identified the table "nyc_data_source"
2. Analyzed the table structure, including all columns and their data types
3. Sampled data from the table to understand the business context
4. Ran additional queries to understand the meaning of coded values (payment_type and ratecodeid)
5. Created a comprehensive metadata.json file containing:
   - Database name and purpose
   - Table name, purpose, location, format, and statistics
   - Column names, data types, and detailed descriptions of their business purpose

The metadata.json file provides a complete overview of the database structure and content. Based on my analysis, this database contains New York City taxi trip data with detailed information about pickups, dropoffs, fares, and various surcharges. The table contains nearly 28 million records of taxi trips with comprehensive information about each journey.